<div style="font-size:12pt; font-weight:bold;">Large Graph Similarity: Example Analysis & Function Tests</div>
<div style="font-size:12pt;">Jonathan H. Morgan, Ph.D.</div>
<div style="font-size:12pt;">26 November 2025</div>

<div style="font-size:10pt; font-weight:bold;">Preamble</div>

In [1]:
# Importing Packages
import os
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Mapping, Any

# Application directory
app_dir = Path(
    "/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin"
)

# Compiled executable
compiled_executable = app_dir / "large_graph_similarity"

# Verify compiled executable exists
assert compiled_executable.exists(), (
    f"Compiled executable not found: {compiled_executable}"
)

# User-facing launcher script
app_executable = app_dir / "run_large_graph_similarity.sh"

# Create launcher script if missing
if not app_executable.exists():

    launcher_text = """#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"

if [ -z "${JULIA_NUM_THREADS:-}" ]; then
    export JULIA_NUM_THREADS=auto
fi

exec "${SCRIPT_DIR}/large_graph_similarity" "$@"
"""

    app_executable.write_text(launcher_text)
    app_executable.chmod(0o755)

# Verify launcher exists
assert app_executable.exists(), (
    f"Launcher not found: {app_executable}"
)

# Input directory (test data)
input_dir = Path(
    "/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data"
)

# Output directory (where we'll write example results)
output_dir = Path(
    "/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs"
)

# Create output directory if needed
output_dir.mkdir(parents=True, exist_ok=True)

# Report configuration
print("Application Directory:")
print(f"  {app_dir}")

print("\nCompiled Executable:")
print(f"  {compiled_executable}")

print("\nLauncher Script:")
print(f"  {app_executable}")

print("\nInput Directory:")
print(f"  {input_dir}")

print("\nOutput Directory:")
print(f"  {output_dir}")

Application Directory:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin

Compiled Executable:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/large_graph_similarity

Launcher Script:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh

Input Directory:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data

Output Directory:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs


<div style="font-size:10pt; font-weight:bold;">Functions</div>

In [2]:
def build_lgs_command(
    app_executable: Path,
    mode: str = "analysis",
    ora_xml_1: Optional[Path] = None,
    ora_xml_2: Optional[Path] = None,
    network_name_1: Optional[str] = None,
    network_name_2: Optional[str] = None,
    ora_leiden: Optional[str] = None,
    edgelist_1: Optional[Path] = None,
    edgelist_2: Optional[Path] = None,
    nodelist_1: Optional[Path] = None,
    nodelist_2: Optional[Path] = None,
    partition_1: Optional[Path] = None,
    partition_2: Optional[Path] = None,
    output_dir: Optional[Path] = None,
    name_1: Optional[str] = None,
    name_2: Optional[str] = None,
    directed: Optional[bool] = True,
    weighted: Optional[bool] = True,
    resolution: Optional[float] = 1.0,
    resolution_sweep: Optional[bool] = False,
    n_resolutions: Optional[int] = 15,
    n_runs: Optional[int] = 5,
    n_iterations: Optional[int] = 10,
    seed: Optional[int] = None,
    verbose: Optional[bool] = True,
    list_functions: bool = False,
    function_help: Optional[str] = None,
) -> List[str]:
    """
    Build the command-line argument list for run_large_graph_similarity.sh.
    Returns a list suitable for subprocess.run(...).
    """
    cmd = [str(app_executable)]

    def add_arg(flag: str, value: Any) -> None:
        if value is None:
            return

        if isinstance(value, bool):
            cmd.extend([flag, "true" if value else "false"])
        else:
            cmd.extend([flag, str(value)])

    # Special flags
    if list_functions:
        cmd.append("--list-functions")

    add_arg("--function-help", function_help)

    # Core mode + IO
    add_arg("--mode", mode)
    add_arg("--ora-xml-1", ora_xml_1)
    add_arg("--ora-xml-2", ora_xml_2)
    add_arg("--network-name-1", network_name_1)
    add_arg("--network-name-2", network_name_2)
    add_arg("--ora-leiden", ora_leiden)

    add_arg("--edgelist-1", edgelist_1)
    add_arg("--edgelist-2", edgelist_2)
    add_arg("--nodelist-1", nodelist_1)
    add_arg("--nodelist-2", nodelist_2)
    add_arg("--partition-1", partition_1)
    add_arg("--partition-2", partition_2)

    add_arg("--output-dir", output_dir)
    add_arg("--name-1", name_1)
    add_arg("--name-2", name_2)

    # Options / tuning
    add_arg("--directed", directed)
    add_arg("--weighted", weighted)
    add_arg("--resolution", resolution)
    add_arg("--resolution-sweep", resolution_sweep)
    add_arg("--n-resolutions", n_resolutions)
    add_arg("--n-runs", n_runs)
    add_arg("--n-iterations", n_iterations)
    add_arg("--seed", seed)
    add_arg("--verbose", verbose)

    return cmd


def run_lgs_command(cmd: Sequence[str]) -> subprocess.CompletedProcess:
    """
    Convenience wrapper to run the command and capture output.
    """
    return subprocess.run(
        cmd,
        text=True,
        capture_output=True,
    )

<div style="font-size:10pt; font-weight:bold;">Basic Function Tests</div>

In [3]:
# Test 1: Documentation Lookup for network_comparator
print("=" * 80)
print("TEST 1: Documentation Lookup")
print("=" * 80)

# 1) List all functions
print("\nRunning: --list-functions")
cmd_list = build_lgs_command(
    app_executable=app_executable,
    list_functions=True,
    verbose=False,
)
print("Command:", " ".join(cmd_list))

result_list = run_lgs_command(cmd_list)
print("\n--- STDOUT ---")
print(result_list.stdout)
print("--- STDERR ---")
print(result_list.stderr)

# 2) Get help for network_comparator
print("\nRunning: --function-help network_comparator")
cmd_help = build_lgs_command(
    app_executable=app_executable,
    function_help="network_comparator",
    verbose=False,
)
print("Command:", " ".join(cmd_help))

result_help = run_lgs_command(cmd_help)
print("\n--- STDOUT ---")
print(result_help.stdout)
print("--- STDERR ---")
print(result_help.stderr)

print("\nTest 1 completed (Python CLI demo)")

TEST 1: Documentation Lookup

Running: --list-functions
Command: /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --list-functions --mode analysis --directed true --weighted true --resolution 1.0 --resolution-sweep false --n-resolutions 15 --n-runs 5 --n-iterations 10 --verbose false

--- STDOUT ---

Available Functions:
  adjusted_rand_index
    Compare two partitions using Adjusted Rand Index

  load_ora_xml
    Load ORA XML metanetwork file

  in_degree
    Calculate in-degree for directed networks

  out_degree
    Calculate out-degree for directed networks

  total_degree
    Calculate total degree

  degree_ratio
    Calculate degree ratios for nodes

  freeman_degree_normalization
    Apply Freeman degree normalization

  local_clustering_coefficient
    Calculate local clustering coefficients

  global_clustering_coefficient
    Calculate global clustering coefficient

In [4]:
# Test 2: Single Network Analysis - Balikatan 2022
print("\n" + "=" * 80)
print("TEST 2: Balikatan 2022 Analysis")
print("=" * 80)

# Output subdirectory for this test
test2_dir = output_dir / "test2_balikatan"
test2_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="analysis",
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - All Communication",
    output_dir=test2_dir,
    name_1="Balikatan_2022",
    directed=True,
    weighted=False,
    resolution=1.0,
    n_runs=3,
    n_iterations=5,
    verbose=True,
)

print("Analyzing Balikatan 2022 network...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run analysis
result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(result.stdout)

print("--- STDERR ---")
print(result.stderr)

# Verify output files exist
expected_files = [
    "Balikatan_2022_global_stats.csv",
    "Balikatan_2022_triad_census.csv",
    "Balikatan_2022_node_measures.csv",
    "Balikatan_2022_feature_vector.csv",
]

print("\nChecking expected output files in:", test2_dir)

all_ok = True

for fname in expected_files:
    fpath = test2_dir / fname

    if fpath.is_file():
        print(f"  ✓ Created: {fname}")
    else:
        print(f"  ✗ Missing: {fname}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "One or more expected output files were not found."
    )

print("\nTest 2 completed successfully")


TEST 2: Balikatan 2022 Analysis
Analyzing Balikatan 2022 network...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode analysis --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --network-name-1 Agent x Agent - All Communication --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/test2_balikatan --name-1 Balikatan_2022 --directed true --weighted false --resolution 1.0 --resolution-sweep false --n-resolutions 15 --n-runs 3 --n-iterations 5 --verbose true

--- STDOUT ---
Loading ORA metanetwork from: /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml
Extracted network 'Agent x Agent - All Communication': 3146 edges, 1347 nodes

=== Runni

In [5]:
# Test 3: Network Comparison - Balikatan vs Pac Rim
print("\n" + "=" * 80)
print("TEST 3: Network Comparison - Balikatan vs Pac Rim")
print("=" * 80)

# Output subdirectory for this test
test3_dir = output_dir / "test3_comparison"
test3_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - All Communication",
    ora_xml_2=input_dir / "Pac Rim Day 1.xml",
    network_name_2="Agent x Agent - All Communication",
    output_dir=test3_dir,
    name_1="Balikatan_2022",
    name_2="PacRim_Day1",
    directed=True,
    weighted=False,
    resolution=1.0,
    n_runs=3,
    n_iterations=5,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Rim Day 1...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(result.stdout)
print("--- STDERR ---")
print(result.stderr)

# Verify expected output files exist
expected_files = [
    # Network 1 analysis
    "Balikatan_2022_global_stats.csv",
    "Balikatan_2022_node_measures.csv",
    "Balikatan_2022_triad_census.csv",

    # Network 2 analysis
    "PacRim_Day1_global_stats.csv",
    "PacRim_Day1_node_measures.csv",
    "PacRim_Day1_triad_census.csv",

    # Comparison results
    "Balikatan_2022_PacRim_Day1_comparison_asinh.csv",
    "Balikatan_2022_PacRim_Day1_comparison_raw.csv",
    "Balikatan_2022_PacRim_Day1_similarity_scores.csv",
    "Balikatan_2022_PacRim_Day1_type_contributions_asinh.csv",
    "Balikatan_2022_PacRim_Day1_type_contributions_raw.csv",
]

print("\nChecking expected output files in:", test3_dir)
all_ok = True
for fname in expected_files:
    fpath = test3_dir / fname
    if fpath.is_file():
        print(f"  ✓ Created: {fname}")
    else:
        print(f"  ✗ Missing: {fname}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("One or more expected output files were not found.")

print("\nTest 3 completed successfully")


TEST 3: Network Comparison - Balikatan vs Pac Rim
Comparing Balikatan 2022 vs Pac Rim Day 1...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Pac Rim Day 1.xml --network-name-1 Agent x Agent - All Communication --network-name-2 Agent x Agent - All Communication --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/test3_comparison --name-1 Balikatan_2022 --name-2 PacRim_Day1 --directed true --weighted false --resolution 1.0 --resolution-sweep false --n-resolutions 15 --n-runs 3 --n-iterations 5 --verbose true

--- STDOUT ---
Loading ORA met

In [6]:
# Test 4: Mixed Comparison – Pac Rim ORA (ORA Leiden) vs Balikatan CSVs
print("\n" + "=" * 80)
print("TEST 4: Mixed Comparison – Pac Rim ORA (ORA Leiden) vs Balikatan CSVs")
print("=" * 80)

# Define input paths
pacrim_ora_path = input_dir / "Pac Rim Day 1.xml"
balikatan_edges = input_dir / "Balikatan_2022_All_Communication_Arcs.csv"
balikatan_nodes = input_dir / "Balikatan_2022_All_Communication_Nodes.csv"
balikatan_part  = input_dir / "All_Comm_Lieden_Group_Assignments.csv"

# Define output directory for this test
test8_dir = output_dir / "test8_mixed_comparison"
test8_dir.mkdir(parents=True, exist_ok=True)

# Names used in outputs
name_1 = "PacRim_Day1_Leiden"
name_2 = "Balikatan_2022"

# Construct CLI command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Pac Rim ORA with ORA Leiden partition
    ora_xml_1=pacrim_ora_path,
    network_name_1="Agent x Agent - All Communication",
    ora_leiden="leiden group",
    name_1=name_1,

    # Network 2: Balikatan from CSVs
    edgelist_2=balikatan_edges,
    nodelist_2=balikatan_nodes,
    partition_2=balikatan_part,
    name_2=name_2,

    # Common options
    output_dir=test8_dir,
    directed=True,
    weighted=True,
    resolution=1.0,
    n_runs=3,
    n_iterations=5,
    verbose=True,
)

print("Running mixed comparison with arguments:")
print("  " + " ".join(cmd))

# Run comparison via CLI
result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(result.stdout)
print("--- STDERR ---")
print(result.stderr)

# Expected output files
expected_files = [
    # Network 1 analysis outputs
    f"{name_1}_global_stats.csv",
    f"{name_1}_node_measures.csv",
    f"{name_1}_triad_census.csv",

    # Network 2 analysis outputs
    f"{name_2}_global_stats.csv",
    f"{name_2}_node_measures.csv",
    f"{name_2}_triad_census.csv",

    # Comparison outputs (name_1_name_2 prefix)
    f"{name_1}_{name_2}_comparison_raw.csv",
    f"{name_1}_{name_2}_comparison_asinh.csv",
    f"{name_1}_{name_2}_similarity_scores.csv",
    f"{name_1}_{name_2}_type_contributions_raw.csv",
    f"{name_1}_{name_2}_type_contributions_asinh.csv",
]

print("\nVerifying expected output files in:", test8_dir)
all_ok = True
for fname in expected_files:
    fpath = test8_dir / fname
    if fpath.is_file():
        print(f"  ✓ Created: {fname}")
    else:
        print(f"  ✗ Missing: {fname}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("One or more expected output files were not found.")

print("\nTest 4 completed successfully")


TEST 4: Mixed Comparison – Pac Rim ORA (ORA Leiden) vs Balikatan CSVs
Running mixed comparison with arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Pac Rim Day 1.xml --network-name-1 Agent x Agent - All Communication --ora-leiden leiden group --edgelist-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_All_Communication_Arcs.csv --nodelist-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_All_Communication_Nodes.csv --partition-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/All_Comm_Lieden_Group_Assignments.csv --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SB

<div style="font-size:10pt; font-weight:bold;">Comparing Pac Sentry's & Pac Rim's Agent x Agent - All Communication Networks to Balikatan 2022</div>

In [ ]:
# Analyzing Balikatan vs Pac Rim Using a Modularity (Resolution) Sweep

# Output subdirectory for this analysis
balikatan_pac_rim_dir = output_dir / "balikatan_pac_rim_dir"
balikatan_pac_rim_dir.mkdir(parents=True, exist_ok=True)

cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - All Communication",
    ora_xml_2=input_dir / "Pac Rim Day 1.xml",
    network_name_2="Agent x Agent - All Communication",
    output_dir=balikatan_pac_rim_dir,
    name_1="Balikatan_2022",
    name_2="PacRim_Day1",
    directed=True,
    weighted=True,
    resolution_sweep=True,   # turn on CHAMP / multi-resolution
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Rim Day 1 with resolution sweep...")
print("Command arguments:")
print("  " + " ".join(cmd))

balikatan_pac_rim_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(balikatan_pac_rim_result.stdout)
print("--- STDERR ---")
print(balikatan_pac_rim_result.stderr)

Comparing Balikatan 2022 vs Pac Rim Day 1 with resolution sweep...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Pac Rim Day 1.xml --network-name-1 Agent x Agent - All Communication --network-name-2 Agent x Agent - All Communication --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_rim_dir --name-1 Balikatan_2022 --name-2 PacRim_Day1 --directed true --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations 10 --seed 42 --verbose true

--- STDOUT ---
Loading ORA metanetwork from:

In [8]:
# Balikatan vs Pac Sentry 2025 using a resolution (modularity) sweep

# Output subdirectory for this analysis
balikatan_pac_sentry_dir = output_dir / "balikatan_pac_sentry_dir"
balikatan_pac_sentry_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Balikatan 2022 ORA metanetwork
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - All Communication",
    name_1="Balikatan_2022",

    # Network 2: Pac Sentry 2025 synthetic network from ORA XML
    ora_xml_2=input_dir / "PAO_Day_6.xml",
    network_name_2="Agent x Agent - All Communication",
    name_2="PacSentry_2025",

    # Common options
    output_dir=balikatan_pac_sentry_dir,
    directed=True,
    weighted=True,        # you used weighted=True in the real analysis block
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Sentry 2025 (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
balikatan_pac_sentry_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(balikatan_pac_sentry_result.stdout)
print("--- STDERR ---")
print(balikatan_pac_sentry_result.stderr)

Comparing Balikatan 2022 vs Pac Sentry 2025 (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/PAO_Day_6.xml --network-name-1 Agent x Agent - All Communication --network-name-2 Agent x Agent - All Communication --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_sentry_dir --name-1 Balikatan_2022 --name-2 PacSentry_2025 --directed true --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations 10 --seed 42 --verbose true

--- STDOUT ---
Loading ORA metanetwork from

<div style="font-size:10pt; font-weight:bold;">Comparing Agent x Agent - Retweeted-By Networks</div>

In [9]:
# Balikatan vs Pac Rim – Agent x Agent - Retweeted-By (resolution sweep)

# Output subdirectory for this analysis
balikatan_pacrim_retweeted_dir = output_dir / "balikatan_pac_rim_retweetedby_dir"
balikatan_pacrim_retweeted_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Balikatan 2022 ORA metanetwork
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - Retweeted-By",
    name_1="Balikatan_2022_RetweetedBy",

    # Network 2: Pac Rim Day 1 ORA metanetwork
    ora_xml_2=input_dir / "Pac Rim Day 1.xml",
    network_name_2="Agent x Agent - Retweeted-By",
    name_2="PacRim_Day1_RetweetedBy",

    # Common options (same sweep params as before)
    output_dir=balikatan_pacrim_retweeted_dir,
    directed=True,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Rim Day 1 – Agent x Agent - Retweeted-By (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
retweeted_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(retweeted_result.stdout)
print("--- STDERR ---")
print(retweeted_result.stderr)

Comparing Balikatan 2022 vs Pac Rim Day 1 – Agent x Agent - Retweeted-By (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Pac Rim Day 1.xml --network-name-1 Agent x Agent - Retweeted-By --network-name-2 Agent x Agent - Retweeted-By --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_rim_retweetedby_dir --name-1 Balikatan_2022_RetweetedBy --name-2 PacRim_Day1_RetweetedBy --directed true --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations 10 --seed 42 --verb

In [10]:
# Balikatan vs Pac Sentry – Agent x Agent - Retweeted-By (resolution sweep)

# Output subdirectory for this analysis
balikatan_pacsentry_retweeted_dir = output_dir / "balikatan_pac_sentry_retweetedby_dir"
balikatan_pacsentry_retweeted_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Balikatan 2022 ORA metanetwork
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - Retweeted-By",
    name_1="Balikatan_2022_RetweetedBy",

    # Network 2: Pac Sentry 2025 ORA metanetwork
    ora_xml_2=input_dir / "PAO_Day_6.xml",
    network_name_2="Agent x Agent - Retweeted-By",
    name_2="PacSentry_2025_RetweetedBy",

    # Common options (same sweep params as before)
    output_dir=balikatan_pacsentry_retweeted_dir,
    directed=True,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Sentry 2025 – Agent x Agent - Retweeted-By (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
retweeted_pacsentry_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(retweeted_pacsentry_result.stdout)
print("--- STDERR ---")
print(retweeted_pacsentry_result.stderr)

Comparing Balikatan 2022 vs Pac Sentry 2025 – Agent x Agent - Retweeted-By (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/PAO_Day_6.xml --network-name-1 Agent x Agent - Retweeted-By --network-name-2 Agent x Agent - Retweeted-By --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_sentry_retweetedby_dir --name-1 Balikatan_2022_RetweetedBy --name-2 PacSentry_2025_RetweetedBy --directed true --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations 10 --seed 42 --

<div style="font-size:10pt; font-weight:bold;">Comparing Agent x Agent - Mentioned-By Networks</div>

In [11]:
# Balikatan vs Pac Rim – Agent x Agent - Mentioned-By (resolution sweep)

# Output subdirectory for this analysis
balikatan_pacrim_mentioned_dir = output_dir / "balikatan_pac_rim_mentionedby_dir"
balikatan_pacrim_mentioned_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Balikatan 2022 ORA metanetwork
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - Mentioned-By",
    name_1="Balikatan_2022_MentionedBy",

    # Network 2: Pac Rim Day 1 ORA metanetwork
    ora_xml_2=input_dir / "Pac Rim Day 1.xml",
    network_name_2="Agent x Agent - Mentioned-By",
    name_2="PacRim_Day1_MentionedBy",

    # Common options (same sweep params as before)
    output_dir=balikatan_pacrim_mentioned_dir,
    directed=True,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Rim Day 1 – Agent x Agent - Mentioned-By (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
mentioned_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(mentioned_result.stdout)
print("--- STDERR ---")
print(mentioned_result.stderr)

Comparing Balikatan 2022 vs Pac Rim Day 1 – Agent x Agent - Mentioned-By (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Pac Rim Day 1.xml --network-name-1 Agent x Agent - Mentioned-By --network-name-2 Agent x Agent - Mentioned-By --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_rim_mentionedby_dir --name-1 Balikatan_2022_MentionedBy --name-2 PacRim_Day1_MentionedBy --directed true --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations 10 --seed 42 --verb

In [12]:
# Balikatan vs Pac Sentry – Agent x Agent - Mentioned-By (resolution sweep)

# Output subdirectory for this analysis
balikatan_pacsentry_mentioned_dir = output_dir / "balikatan_pac_sentry_mentionedby_dir"
balikatan_pacsentry_mentioned_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Balikatan 2022 ORA metanetwork
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Agent x Agent - Mentioned-By",
    name_1="Balikatan_2022_MentionedBy",

    # Network 2: Pac Sentry 2025 ORA metanetwork
    ora_xml_2=input_dir / "PAO_Day_6.xml",
    network_name_2="Agent x Agent - Mentioned-By",
    name_2="PacSentry_2025_MentionedBy",

    # Common options (same sweep params as before)
    output_dir=balikatan_pacsentry_mentioned_dir,
    directed=True,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Sentry 2025 – Agent x Agent - Mentioned-By (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
mentioned_pacsentry_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(mentioned_pacsentry_result.stdout)
print("--- STDERR ---")
print(mentioned_pacsentry_result.stderr)

Comparing Balikatan 2022 vs Pac Sentry 2025 – Agent x Agent - Mentioned-By (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/PAO_Day_6.xml --network-name-1 Agent x Agent - Mentioned-By --network-name-2 Agent x Agent - Mentioned-By --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_sentry_mentionedby_dir --name-1 Balikatan_2022_MentionedBy --name-2 PacSentry_2025_MentionedBy --directed true --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations 10 --seed 42 --

<div style="font-size:10pt; font-weight:bold;">Comparing Hashtag x Hashtag - Co-Occurrence Networks</div>

In [13]:
# Balikatan vs Pac Rim – Hashtag x Hashtag - Co-Occurrence (resolution sweep)

# Output subdirectory for this analysis
balikatan_pacrim_hashtag_dir = output_dir / "balikatan_pac_rim_hashtag_cooccurrence_dir"
balikatan_pacrim_hashtag_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Balikatan 2022 ORA metanetwork
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Hashtag x Hashtag - Co-Occurrence",
    name_1="Balikatan_2022_HashtagCoOcc",

    # Network 2: Pac Rim Day 1 ORA metanetwork
    ora_xml_2=input_dir / "Pac Rim Day 1.xml",
    network_name_2="Hashtag x Hashtag - Co-Occurrence",
    name_2="PacRim_Day1_HashtagCoOcc",

    # Common options (same sweep params as prior runs)
    output_dir=balikatan_pacrim_hashtag_dir,
    directed=False,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Rim Day 1 – Hashtag x Hashtag - Co-Occurrence (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
hashtag_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(hashtag_result.stdout)
print("--- STDERR ---")
print(hashtag_result.stderr)

Comparing Balikatan 2022 vs Pac Rim Day 1 – Hashtag x Hashtag - Co-Occurrence (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Pac Rim Day 1.xml --network-name-1 Hashtag x Hashtag - Co-Occurrence --network-name-2 Hashtag x Hashtag - Co-Occurrence --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_rim_hashtag_cooccurrence_dir --name-1 Balikatan_2022_HashtagCoOcc --name-2 PacRim_Day1_HashtagCoOcc --directed false --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-ite

In [14]:
# Balikatan vs Pac Sentry – Hashtag x Hashtag - Co-Occurrence (resolution sweep)

# Output subdirectory for this analysis
balikatan_pacsentry_hashtag_dir = output_dir / "balikatan_pac_sentry_hashtag_cooccurrence_dir"
balikatan_pacsentry_hashtag_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: Balikatan 2022 ORA metanetwork
    ora_xml_1=input_dir / "Balikatan_2022_Processed.xml",
    network_name_1="Hashtag x Hashtag - Co-Occurrence",
    name_1="Balikatan_2022_HashtagCoOcc",

    # Network 2: Pac Sentry 2025 ORA metanetwork
    ora_xml_2=input_dir / "PAO_Day_6.xml",
    network_name_2="Hashtag x Hashtag - Co-Occurrence",
    name_2="PacSentry_2025_HashtagCoOcc",

    # Common options (same sweep params as prior runs)
    output_dir=balikatan_pacsentry_hashtag_dir,
    directed=False,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing Balikatan 2022 vs Pac Sentry 2025 – Hashtag x Hashtag - Co-Occurrence (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
hashtag_pacsentry_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(hashtag_pacsentry_result.stdout)
print("--- STDERR ---")
print(hashtag_pacsentry_result.stderr)

Comparing Balikatan 2022 vs Pac Sentry 2025 – Hashtag x Hashtag - Co-Occurrence (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/PAO_Day_6.xml --network-name-1 Hashtag x Hashtag - Co-Occurrence --network-name-2 Hashtag x Hashtag - Co-Occurrence --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/balikatan_pac_sentry_hashtag_cooccurrence_dir --name-1 Balikatan_2022_HashtagCoOcc --name-2 PacSentry_2025_HashtagCoOcc --directed false --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n

<div style="font-size:10pt; font-weight:bold;">Cross Platform (Twitters vs. Telegram) Analysis</div>

In [ ]:
# BK12 Telegram vs Balikatan – User x URL x User vs. Agent x URL x Agent

# Output subdirectory for this comparison
telegram_balikatan_url_dir = output_dir / "telegram_balikatan_url_comparison"
telegram_balikatan_url_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1: BK12 Telegram Summary Network
    ora_xml_1=input_dir / "BK12_Telegram_Summary_Network.xml",
    network_name_1="User x Url x User",
    name_1="BK12_Telegram_User_URL",

    # Network 2: Balikatan 2022 ORA metanetwork
    ora_xml_2=input_dir / "Balikatan_2022_Processed.xml",
    network_name_2="Agent x Url x Agent",
    name_2="Balikatan_2022_Agent_URL",

    # Common options
    output_dir=telegram_balikatan_url_dir,
    directed=True,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Comparing BK12 Telegram User x URL vs Balikatan Agent x URL (resolution sweep)...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
telegram_balikatan_url_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(telegram_balikatan_url_result.stdout)
print("--- STDERR ---")
print(telegram_balikatan_url_result.stderr)

Comparing BK12 Telegram User x URL vs Balikatan Agent x URL (resolution sweep)...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/BK12_Telegram_Summary_Network.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Balikatan_2022_Processed.xml --network-name-1 User x Url x User --network-name-2 Agent x Url x Agent --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/telegram_balikatan_url_comparison --name-1 BK12_Telegram_User_URL --name-2 Balikatan_2022_Agent_URL --directed true --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations 10 --seed 42 --verbose true

--- STDOUT

<div style="font-size:10pt; font-weight:bold;">Scale Test: Comparing the Full Marvel Universe, Hero Projection to a Reduced Version</div>

In [ ]:
# Scale Test: Marvel Universe Hero Projection Comparison

print("\n" + "=" * 80)
print("SCALE TEST: Marvel Universe Hero Projection Comparison")
print("=" * 80)

# Output subdirectory for this comparison
marvel_scale_dir = output_dir / "marvel_scale_test"
marvel_scale_dir.mkdir(parents=True, exist_ok=True)

# Build command
cmd = build_lgs_command(
    app_executable=app_executable,
    mode="comparison",

    # Network 1
    ora_xml_1=input_dir / "Marvel_Universe_Hero_Projection.xml",
    network_name_1="Heroes x Heroes - Shared Comic Books",
    name_1="Marvel_Heroes_1",

    # Network 2
    ora_xml_2=input_dir / "Marvel_Universe_Hero_Projection_Reduced.xml",
    network_name_2="Heroes x Heroes - Shared Comic Books",
    name_2="Marvel_Heroes_2",

    # Common options
    output_dir=marvel_scale_dir,
    directed=False,
    weighted=True,
    resolution_sweep=True,
    n_resolutions=25,
    n_runs=10,
    n_iterations=10,
    seed=42,
    verbose=True,
)

print("Running Marvel Universe scale comparison...")
print("Command arguments:")
print("  " + " ".join(cmd))

# Run comparison
marvel_scale_result = run_lgs_command(cmd)

print("\n--- STDOUT ---")
print(marvel_scale_result.stdout)

print("--- STDERR ---")
print(marvel_scale_result.stderr)


SCALE TEST: Marvel Universe Hero Projection Comparison
Running Marvel Universe scale comparison...
Command arguments:
  /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/julia_env/build/Large_Graph_Similarity_app/bin/run_large_graph_similarity.sh --mode comparison --ora-xml-1 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Marvel_Universe_Hero_Projection.xml --ora-xml-2 /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/Marvel_Universe_Hero_Projection_Reduced.xml --network-name-1 Heroes x Heroes - Shared Comic Books --network-name-2 Heroes x Heroes - Shared Comic Books --output-dir /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Example_Outputs/marvel_scale_test --name-1 Marvel_Heroes_1 --name-2 Marvel_Heroes_2 --directed false --weighted true --resolution 1.0 --resolution-sweep true --n-resolutions 25 --n-runs 10 --n-iterations